# Phase 2: Feature Engineering

**Goal**: Extract production-ready forensic features from raw LogFile/UsnJrnl fields

**Input**:
- `data/processed/Phase 1 - Data Cleaning/training_events.csv` (373,114 events with labels)
- `data/validation/processed/[dataset]_events.csv` (5 files without labels)

**Output**:
- `data/processed/Phase 2 - Feature Engineering/training_features.csv`
- `data/processed/Phase 2 - Feature Engineering/validation/[dataset]_features.csv` (5 files)

**Feature Categories**:
1. **Forensic Patterns**: Zero nanoseconds, time reversal, basic info changes
2. **Cross-Artifact Validation**: Evidence from both LogFile and UsnJrnl
3. **Temporal Patterns**: Event frequency and clustering
4. **File Characteristics**: File types, paths, extensions
5. **Timestamp Changes**: Before/after timestamp analysis




## Imports and Setup

In [15]:
# Cell 1: Imports
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("Phase 2: Feature Engineering")
print("="*80)


Phase 2: Feature Engineering


## Directory Configuration

**Input directories**:
- Training events from Phase 1
- Validation events from Phase 1 (without ground truth labels)

**Output directories**:
- `data/processed/Phase 2 - Feature Engineering/` for training features
- `data/validation/processed/phase 2` for validation features


In [16]:
# Cell 2: Directory Configuration

BASE_DIR = Path('/Users/soni/Github/Digital-Detectives_Thesis')

# Input paths (from Phase 1)
TRAINING_EVENTS = BASE_DIR / 'data/processed/Phase 1 - Data Cleaning/training_events.csv'
VALIDATION_EVENTS_DIR = BASE_DIR / 'data/validation/processed'

# Output paths
OUTPUT_DIR = BASE_DIR / 'data/processed/Phase 2 - Feature Engineering'
VALIDATION_OUTPUT_DIR = BASE_DIR / 'data/validation/processed/phase 2'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VALIDATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Directories configured")
print(f"Training input: {TRAINING_EVENTS}")
print(f"Validation input: {VALIDATION_EVENTS_DIR}")
print(f"Output: {OUTPUT_DIR}")


Directories configured
Training input: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 1 - Data Cleaning/training_events.csv
Validation input: /Users/soni/Github/Digital-Detectives_Thesis/data/validation/processed
Output: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 2 - Feature Engineering


## Data Cleaning and Standardization

**Step 1: Drop Unnecessary Columns**

The following columns are dropped because they don't contribute to timestomping detection:
- `Carving Flag`: Empty in all records
- `SourceInfo`: All values are "Normal" (no variance)
- `Target VCN`: NTFS disk allocation detail (not relevant for timestamp manipulation)
- `Cluster Index`: NTFS disk location (not relevant for timestamp manipulation)
- `Redo`: NTFS transaction internals (not relevant)
- `lsn/usn`: Duplicate of LSN/USN columns (added during Suspicious CSV merge)

**Step 2: Standardize Column Names**

LogFile and UsnJrnl have different column names for similar data:
- `File/Directory Name` → `filename`
- `Full Path` / `FullPath` → `full_path`
- `EventTime(UTC+8)` / `TimeStamp(UTC+8)` → `event_time`
- `Event` / `EventInfo` → `event_type`

**Step 3: Reorder Columns**

Place `dataset` column first (leftmost) for easy filtering and grouping.


In [17]:
# Cell 3: Data Cleaning Function

def load_and_clean_events(file_path, is_training=True):
    """
    Load events CSV and perform initial cleaning.
    
    Steps:
    1. Load CSV
    2. Drop unnecessary columns
    3. Standardize column names
    4. Reorder columns (dataset first)
    
    Parameters:
        file_path: Path to events CSV
        is_training: Whether this is training data (has ground truth labels)
    
    Returns:
        Cleaned DataFrame
    """
    print(f"\nLoading: {file_path.name}")
    df = pd.read_csv(file_path, low_memory=False)
    
    print(f"  Original shape: {df.shape}")
    print(f"  Original columns: {len(df.columns)}")
    
    # Drop unnecessary columns
    drop_cols = [
        'Carving Flag',      # Empty
        'SourceInfo',        # All "Normal"
        'Target VCN',        # NTFS disk allocation (not relevant)
        'Cluster Index',     # NTFS disk location (not relevant)
        'Redo',              # NTFS transaction detail (not relevant)
        'lsn/usn'            # Duplicate of LSN/USN
    ]
    
    # Only drop columns that exist
    drop_cols = [col for col in drop_cols if col in df.columns]
    df = df.drop(columns=drop_cols)
    
    print(f"  Dropped {len(drop_cols)} columns")
    
    # Standardize column names
    rename_map = {
        'File/Directory Name': 'filename',
        'Full Path': 'full_path_lf',
        'FullPath': 'full_path_usn',
        'EventTime(UTC+8)': 'event_time_lf',
        'TimeStamp(UTC+8)': 'event_time_usn',
        'Event': 'event_type_lf',
        'EventInfo': 'event_type_usn',
        'Detail': 'detail_lf',
        'FileAttribute': 'file_attribute',
        'FileReferenceNumber': 'file_ref_number',
        'ParentFileReferenceNumber': 'parent_file_ref_number'
    }
    
    # Only rename columns that exist
    rename_map = {k: v for k, v in rename_map.items() if k in df.columns}
    df = df.rename(columns=rename_map)
    
    # Create unified columns from source-specific ones
    if 'full_path_lf' in df.columns and 'full_path_usn' in df.columns:
        df['full_path'] = df['full_path_lf'].fillna(df['full_path_usn'])
        df = df.drop(columns=['full_path_lf', 'full_path_usn'])
    elif 'full_path_usn' in df.columns:
        df['full_path'] = df['full_path_usn']
        df = df.drop(columns=['full_path_usn'])
    
    if 'event_time_lf' in df.columns and 'event_time_usn' in df.columns:
        df['event_time'] = df['event_time_lf'].fillna(df['event_time_usn'])
        df = df.drop(columns=['event_time_lf', 'event_time_usn'])
    elif 'event_time_usn' in df.columns:
        df['event_time'] = df['event_time_usn']
        df = df.drop(columns=['event_time_usn'])
    
    if 'event_type_lf' in df.columns and 'event_type_usn' in df.columns:
        df['event_type'] = df['event_type_lf'].fillna(df['event_type_usn'])
        df = df.drop(columns=['event_type_lf', 'event_type_usn'])
    elif 'event_type_usn' in df.columns:
        df['event_type'] = df['event_type_usn']
        df = df.drop(columns=['event_type_usn'])
    
    # Reorder columns: dataset first
    cols = df.columns.tolist()
    if 'dataset' in cols:
        cols.remove('dataset')
        cols = ['dataset'] + cols
        df = df[cols]
    
    print(f"  Final shape: {df.shape}")
    print(f"  Final columns: {len(df.columns)}")
    
    return df

# Load training data
training_df = load_and_clean_events(TRAINING_EVENTS, is_training=True)

print(f"\nTraining data loaded:")
print(f"  Total events: {len(training_df):,}")
print(f"  Datasets: {training_df['dataset'].nunique()}")
if 'is_suspicious' in training_df.columns:
    print(f"  Suspicious events: {training_df['is_suspicious'].sum():,}")
print(f"  Source artifacts: {training_df['source_artifact'].value_counts().to_dict()}")



Loading: training_events.csv
  Original shape: (373114, 28)
  Original columns: 28
  Dropped 6 columns
  Final shape: (373114, 19)
  Final columns: 19

Training data loaded:
  Total events: 373,114
  Datasets: 22
  Suspicious events: 288
  Source artifacts: {'usnjrnl': 366284, 'logfile': 6830}


## Feature Engineering

### 1. Forensic Pattern Features

These features detect known timestomping signatures based on Oh et al. (2024) methodology.

#### 1.1 Zero in 100-Nanoseconds

**What it detects**: Windows `SetFileTime()` API signature

**Why it's suspicious**: The `SetFileTime()` API sets the nanoseconds field to zero. Natural filesystem operations preserve nanoseconds from the system clock. Zero nanoseconds indicate programmatic timestamp manipulation.

**Pattern in LogFile Detail field**:
- `"Zero in 100-nanoseconds"`
- `"Zero in CreationTime's 100-nanoseconds"`
- `"Zero in ModifiedTime's 100-nanoseconds"`

**Features created**:
- `zero_in_nanoseconds`: Boolean - ANY timestamp has zero nanoseconds
- `zero_in_creation`: Boolean - CreationTime has zero nanoseconds
- `zero_in_modified`: Boolean - ModifiedTime has zero nanoseconds
- `zero_in_mft_modified`: Boolean - MFTModifiedTime has zero nanoseconds
- `zero_in_accessed`: Boolean - AccessedTime has zero nanoseconds

#### 1.2 Time Reversal Event

**What it detects**: NTFS detected a timestamp changed to the past

**Why it's suspicious**: The NTFS filesystem logs when $SI (Standard Information) timestamps are set to earlier values than previously recorded. This is a strong indicator of timestomping.

**Pattern in Event field**: `"Time Reversal Event"`

**Feature created**: `time_reversal_event`: Boolean

#### 1.3 Basic Info Change

**What it detects**: UsnJrnl recorded a BASIC_INFO_CHANGE event

**Why it's suspicious**: The `BASIC_INFO_CHANGE` reason code in UsnJrnl indicates file metadata (including timestamps) was modified. When combined with other indicators, this suggests timestamp manipulation.

**Pattern in EventInfo field**: `"Basic_Info_Change"`

**Feature created**: `basic_info_changed`: Boolean

#### 1.4 Additional Forensic Patterns

**Timestamp Changed to Past**: Explicit indication in Detail field  
**Using Another File's Timestamp**: Attacker copied timestamps from another file  
**Update Resident Value**: NTFS transaction type for metadata updates

**Features created**:
- `update_resident_value`: Boolean
- `timestamp_to_past`: Boolean
- `using_another_timestamp`: Boolean


In [18]:
# Cell 4: Forensic Pattern Features

def extract_forensic_patterns(df):
    """
    Extract forensic pattern features from raw LogFile/UsnJrnl fields.
    All features are extractable from production data (no ground truth needed).
    
    Features created: 11 total
    - zero_in_nanoseconds (+ 4 specific timestamp variants)
    - time_reversal_event
    - basic_info_changed
    - update_resident_value
    - timestamp_to_past
    - using_another_timestamp
    """
    print("\nExtracting forensic pattern features...")
    
    # 1. ZERO IN NANOSECONDS (from LogFile Detail field)
    df['zero_in_nanoseconds'] = df['detail_lf'].fillna('').str.contains(
        'Zero in.*100-nanoseconds',
        case=False,
        na=False,
        regex=True
    )
    
    # Which specific timestamp has zero nanoseconds?
    df['zero_in_creation'] = df['detail_lf'].fillna('').str.contains(
        'CreationTime.*Zero in.*100-nanoseconds',
        case=False,
        na=False,
        regex=True
    )
    
    df['zero_in_modified'] = df['detail_lf'].fillna('').str.contains(
        'ModifiedTime.*Zero in.*100-nanoseconds',
        case=False,
        na=False,
        regex=True
    )
    
    df['zero_in_mft_modified'] = df['detail_lf'].fillna('').str.contains(
        'MFTModifiedTime.*Zero in.*100-nanoseconds',
        case=False,
        na=False,
        regex=True
    )
    
    df['zero_in_accessed'] = df['detail_lf'].fillna('').str.contains(
        'AccessedTime.*Zero in.*100-nanoseconds',
        case=False,
        na=False,
        regex=True
    )
    
    # 2. TIME REVERSAL EVENT
    df['time_reversal_event'] = df['event_type'].fillna('').str.contains(
        'Time Reversal',
        case=False,
        na=False
    )
    
    # 3. BASIC INFO CHANGE
    df['basic_info_changed'] = df['event_type'].fillna('').str.contains(
        'Basic_Info_Change',
        case=False,
        na=False
    )
    
    # 4. UPDATE RESIDENT VALUE
    df['update_resident_value'] = df['event_type'].fillna('').str.contains(
        'Update',
        case=False,
        na=False
    )
    
    # 5. TIMESTAMP CHANGED TO PAST
    df['timestamp_to_past'] = df['detail_lf'].fillna('').str.contains(
        'timestamp changed to past',
        case=False,
        na=False
    )
    
    # 6. USING ANOTHER FILE'S TIMESTAMP
    df['using_another_timestamp'] = df['detail_lf'].fillna('').str.contains(
        'Using another file',
        case=False,
        na=False
    )
    
    print(f"  zero_in_nanoseconds: {df['zero_in_nanoseconds'].sum():,} events")
    print(f"  time_reversal_event: {df['time_reversal_event'].sum():,} events")
    print(f"  basic_info_changed: {df['basic_info_changed'].sum():,} events")
    print(f"  timestamp_to_past: {df['timestamp_to_past'].sum():,} events")
    print(f"  using_another_timestamp: {df['using_another_timestamp'].sum():,} events")
    
    return df


### 2. Cross-Artifact Validation Features

**Concept**: Events detected by BOTH LogFile and UsnJrnl have higher confidence

**Why it matters**: Oh et al. (2024) found that cross-artifact validation significantly improves detection accuracy. When the same file appears in both LogFile and UsnJrnl with timestamp-related events, it's stronger evidence of manipulation.

**CRITICAL Implementation Detail**: Cross-artifact validation must be checked WITHIN each dataset, not globally. A file appearing in dataset "01-PE" LogFile and dataset "02-PE" UsnJrnl should NOT be flagged as cross-artifact validated.

**Features created**:
- `has_logfile_evidence`: Boolean - This FILE appears in LogFile within same dataset
- `has_usnjrnl_evidence`: Boolean - This FILE appears in UsnJrnl within same dataset
- `cross_artifact_validation_score`: Float
  - 0.5 = Detected by one source only
  - 1.0 = Detected by BOTH sources (high confidence)

**Example**:
Dataset: LoneWolf File: DeathToll.jpg
- 2 LogFile events (Time Reversal)
- 6 UsnJrnl events (Basic_Info_Change)
- ALL 8 events get cross_artifact_validation_score = 1.0

In [19]:
# Cell 5: Cross-Artifact Validation Features

def add_cross_artifact_validation(df):
    """
    Add cross-artifact validation features.
    
    CRITICAL: Check within EACH dataset separately!
    A file in dataset1 LogFile + dataset2 UsnJrnl should NOT be cross-validated.
    
    Features created: 3 total
    - has_logfile_evidence
    - has_usnjrnl_evidence
    - cross_artifact_validation_score
    """
    print("\nAdding cross-artifact validation features...")
    
    # Initialize columns
    df['has_logfile_evidence'] = False
    df['has_usnjrnl_evidence'] = False
    
    # For EACH dataset, find files that appear in both sources
    for dataset_name in df['dataset'].unique():
        dataset_mask = df['dataset'] == dataset_name
        
        # Files in LogFile for THIS dataset only
        files_in_lf = set(
            df[(dataset_mask) & (df['source_artifact'] == 'logfile')]['filename'].dropna()
        )
        
        # Files in UsnJrnl for THIS dataset only
        files_in_usn = set(
            df[(dataset_mask) & (df['source_artifact'] == 'usnjrnl')]['filename'].dropna()
        )
        
        # Files in both (cross-artifact validated)
        files_in_both = files_in_lf & files_in_usn
        
        # Update flags for this dataset
        df.loc[dataset_mask & (df['filename'].isin(files_in_lf)), 'has_logfile_evidence'] = True
        df.loc[dataset_mask & (df['filename'].isin(files_in_usn)), 'has_usnjrnl_evidence'] = True
        
        print(f"  {dataset_name:20s} LF: {len(files_in_lf):4d} files | USN: {len(files_in_usn):4d} files | Both: {len(files_in_both):4d} files")
    
    # Calculate cross-artifact validation score
    # 0.5 = one source, 1.0 = both sources
    df['cross_artifact_validation_score'] = (
        df['has_logfile_evidence'].astype(int) * 0.5 +
        df['has_usnjrnl_evidence'].astype(int) * 0.5
    )
    
    print(f"\n  Events with both sources (score=1.0): {(df['cross_artifact_validation_score'] == 1.0).sum():,}")
    print(f"  Events with one source (score=0.5): {(df['cross_artifact_validation_score'] == 0.5).sum():,}")
    
    return df


In [20]:
# Cell 6: Temporal Features

def add_temporal_features(df):
    """
    Add temporal pattern features.
    
    Features created: 4 total
    - event_count_per_file
    - events_in_1min_window
    - events_in_5min_window
    - event_datetime
    """
    print("\nAdding temporal features...")
    
    # Convert event time to datetime
    df['event_datetime'] = pd.to_datetime(df['event_time'], errors='coerce')
    
    # 1. EVENT COUNT PER FILE (within same dataset)
    df['event_count_per_file'] = df.groupby(['dataset', 'filename'])['filename'].transform('count')
    
    # 2. EVENTS IN TIME WINDOWS
    # Sort by dataset, filename, and time
    df = df.sort_values(['dataset', 'filename', 'event_datetime'])
    
    def count_events_in_window(group, window_minutes):
        """Count events within ±window_minutes for each event in group"""
        if len(group) <= 1:
            return pd.Series([1] * len(group), index=group.index)
        
        times = group['event_datetime'].values
        counts = []
        
        for i, time in enumerate(times):
            if pd.isna(time):
                counts.append(1)
                continue
            
            window_start = time - pd.Timedelta(minutes=window_minutes)
            window_end = time + pd.Timedelta(minutes=window_minutes)
            
            # Count events in window
            count = ((times >= window_start) & (times <= window_end)).sum()
            counts.append(count)
        
        return pd.Series(counts, index=group.index)
    
    # Apply window counting per dataset+filename group
    print("  Calculating 1-minute window counts...")
    df['events_in_1min_window'] = df.groupby(['dataset', 'filename'], group_keys=False).apply(
        lambda g: count_events_in_window(g, 1)
    )
    
    print("  Calculating 5-minute window counts...")
    df['events_in_5min_window'] = df.groupby(['dataset', 'filename'], group_keys=False).apply(
        lambda g: count_events_in_window(g, 5)
    )
    
    print(f"  Max events per file: {df['event_count_per_file'].max()}")
    print(f"  Max events in 1min window: {df['events_in_1min_window'].max()}")
    print(f"  Max events in 5min window: {df['events_in_5min_window'].max()}")
    
    return df


### 4. File Characteristic Features

**Concept**: Certain file types are more likely to be timestomped by attackers

**Why it matters**: 
- Executables (.exe, .dll, .sys) are often timestomped to evade detection
- Documents (.pdf, .doc) may be timestomped for anti-forensics
- System files are rarely legitimately timestomped
- File location and naming patterns can indicate attacker behavior

**Features created**:
- `is_executable`: Boolean - Executable file types (.exe, .dll, .sys, .bat, .ps1, .vbs, .js)
- `is_document`: Boolean - Document file types (.doc, .docx, .pdf, .txt, .xls, .ppt, etc.)
- `is_image`: Boolean - Image file types (.jpg, .png, .gif, .bmp, etc.)
- `is_archive`: Boolean - Archive file types (.zip, .rar, .7z, .tar, .gz)
- `is_system_file`: Boolean - File located in Windows system directories
- `path_depth`: Integer - Number of directory levels (e.g., `\Windows\System32\file.dll` = 2)
- `filename_length`: Integer - Length of filename
- `file_extension`: String - File extension for analysis

**Detection patterns**:
- Executables in unusual locations = Suspicious
- Deep path depths may indicate hiding
- Very short/long filenames may indicate automated tools


In [21]:
# Cell 7: File Characteristic Features

def add_file_characteristics(df):
    """
    Add file characteristic features.
    
    Features created: 8 total
    - is_executable, is_document, is_image, is_archive
    - is_system_file
    - path_depth
    - filename_length
    - file_extension
    """
    print("\nAdding file characteristic features...")
    
    # File type indicators
    df['is_executable'] = df['filename'].fillna('').str.lower().str.endswith(
        ('.exe', '.dll', '.sys', '.bat', '.cmd', '.ps1', '.vbs', '.js')
    )
    
    df['is_document'] = df['filename'].fillna('').str.lower().str.endswith(
        ('.doc', '.docx', '.pdf', '.txt', '.xls', '.xlsx', '.ppt', '.pptx', '.rtf')
    )
    
    df['is_image'] = df['filename'].fillna('').str.lower().str.endswith(
        ('.jpg', '.jpeg', '.png', '.gif', '.bmp', '.tif', '.tiff')
    )
    
    df['is_archive'] = df['filename'].fillna('').str.lower().str.endswith(
        ('.zip', '.rar', '.7z', '.tar', '.gz', '.bz2')
    )
    
    df['is_system_file'] = df['full_path'].fillna('').str.contains(
        r'\\Windows\\|\\System32\\|\\SysWOW64\\',
        case=False,
        na=False,
        regex=True
    )
    
    # Path characteristics
    df['path_depth'] = df['full_path'].fillna('').str.count(r'\\\\')
    df['filename_length'] = df['filename'].fillna('').str.len()
    
    # File extension
    df['file_extension'] = df['filename'].fillna('').str.extract(r'\.([^.]+)$')[0].fillna('')
    
    print(f"  Executables: {df['is_executable'].sum():,}")
    print(f"  Documents: {df['is_document'].sum():,}")
    print(f"  Images: {df['is_image'].sum():,}")
    print(f"  System files: {df['is_system_file'].sum():,}")
    
    return df


### 5. Timestamp Change Features

**Concept**: Parse actual timestamp changes from LogFile Detail field

**Why it matters**: The Detail field contains before/after timestamps, allowing us to analyze:
- How far back in time the timestamp was changed
- Patterns in timestamp changes (e.g., always setting to midnight)

**Pattern in Detail field**:
"CreationTime : 2018-04-06 20:35:25 -> 2018-04-05 10:13:47(Zero in 100-nanoseconds)" Before: 2018-04-06 20:35:25 After: 2018-04-05 10:13:47 Change: -34.2 hours (moved to past)

**Features created**:
- `timestamp_before`: String - Original timestamp
- `timestamp_after`: String - New timestamp
- `timestamp_before_dt`: Datetime - Parsed original timestamp
- `timestamp_after_dt`: Datetime - Parsed new timestamp
- `timestamp_change_seconds`: Float - Time difference in seconds (negative = moved to past)

**Detection pattern**:
- Large negative values = Timestamp moved far to the past
- Specific values (e.g., -86400 = exactly 1 day) may indicate automated tools


In [22]:
# Cell 8: Timestamp Change Features

def parse_timestamp_changes(df):
    """
    Parse before/after timestamps from LogFile Detail field.
    
    Pattern: "CreationTime : 2018-04-06 20:35:25 -> 2018-04-05 10:13:47"
    
    Features created: 5 total
    - timestamp_before, timestamp_after (strings)
    - timestamp_before_dt, timestamp_after_dt (datetimes)
    - timestamp_change_seconds (float, negative = moved to past)
    """
    print("\nParsing timestamp changes from Detail field...")
    
    # Extract before and after timestamps
    timestamp_pattern = r':\s*([^-]+?)\s*->\s*([^(]+)'
    
    df[['timestamp_before', 'timestamp_after']] = df['detail_lf'].str.extract(
        timestamp_pattern
    )
    
    # Clean up whitespace
    df['timestamp_before'] = df['timestamp_before'].str.strip()
    df['timestamp_after'] = df['timestamp_after'].str.strip()
    
    # Convert to datetime
    df['timestamp_before_dt'] = pd.to_datetime(df['timestamp_before'], errors='coerce')
    df['timestamp_after_dt'] = pd.to_datetime(df['timestamp_after'], errors='coerce')
    
    # Calculate time difference (negative = timestamp moved to past)
    df['timestamp_change_seconds'] = (
        df['timestamp_after_dt'] - df['timestamp_before_dt']
    ).dt.total_seconds()
    
    parsed_count = df['timestamp_before'].notna().sum()
    moved_to_past = (df['timestamp_change_seconds'] < 0).sum()
    
    print(f"  Parsed timestamp changes: {parsed_count:,}")
    print(f"  Changes to past (negative seconds): {moved_to_past:,}")
    if parsed_count > 0:
        print(f"  Mean change (seconds): {df['timestamp_change_seconds'].mean():.0f}")
        print(f"  Min change (most past): {df['timestamp_change_seconds'].min():.0f}")
        print(f"  Max change (most future): {df['timestamp_change_seconds'].max():.0f}")
    
    return df


## Process Training Data

Apply all feature engineering steps to training data in sequence.

**Steps**:
1. Forensic patterns (11 features)
2. Cross-artifact validation (3 features)
3. Temporal patterns (4 features)
4. File characteristics (8 features)
5. Timestamp changes (5 features)

**Total**: 31 new features created


In [23]:
# Cell 9: Process Training Data

print("\n" + "="*80)
print("PROCESSING TRAINING DATA")
print("="*80)

# Apply all feature engineering steps
training_df = extract_forensic_patterns(training_df)
training_df = add_cross_artifact_validation(training_df)
training_df = add_temporal_features(training_df)
training_df = add_file_characteristics(training_df)
training_df = parse_timestamp_changes(training_df)

# Save training features
output_path = OUTPUT_DIR / 'training_features.csv'
training_df.to_csv(output_path, index=False)

print(f"\n" + "="*80)
print(f"TRAINING FEATURES SAVED")
print(f"="*80)
print(f"Output: {output_path}")
print(f"Shape: {training_df.shape}")
print(f"Size: {output_path.stat().st_size / 1024 / 1024:.2f} MB")
print(f"\nTotal columns: {len(training_df.columns)}")
if 'is_suspicious' in training_df.columns:
    print(f"Suspicious events: {training_df['is_suspicious'].sum():,} / {len(training_df):,}")



PROCESSING TRAINING DATA

Extracting forensic pattern features...
  zero_in_nanoseconds: 2,627 events
  time_reversal_event: 6,824 events
  basic_info_changed: 366,284 events
  timestamp_to_past: 0 events
  using_another_timestamp: 0 events

Adding cross-artifact validation features...
  01-PE                LF:  909 files | USN: 5925 files | Both:  904 files
  02-PE                LF:   91 files | USN: 8681 files | Both:   91 files
  03-PE                LF:   94 files | USN: 8670 files | Both:   93 files
  04-PE                LF:   79 files | USN:  854 files | Both:   78 files
  05-PE                LF:  142 files | USN:  902 files | Both:  142 files
  06-PE                LF:  142 files | USN:  896 files | Both:  142 files
  07-PE                LF:  185 files | USN: 8794 files | Both:  185 files
  08-PE                LF:  185 files | USN: 8797 files | Both:  185 files
  09-PE                LF:  281 files | USN: 8894 files | Both:  280 files
  10-PE                LF:  282 files

## Process Validation Data

Apply the SAME feature engineering pipeline to validation datasets.

**Critical**: Validation datasets have NO ground truth labels in the events CSV. Labels are stored separately in `ground_truth/[dataset]_ground_truth.csv` for Phase 4 evaluation.

**Validation datasets**:
1. LoneWolf (12 files)
2. 02-APT19 (1 file)
3. 09-APT40 (1 file)
4. 12-Kimsuky (3 files)
5. 13-Winnti731 (1 file)

**Total**: 18 validation files, all with "zero nanoseconds" pattern


In [24]:
# Cell 10: Process Validation Data

print("\n" + "="*80)
print("PROCESSING VALIDATION DATA")
print("="*80)

validation_datasets = ['LoneWolf', '02-APT19', '09-APT40', '12-Kimsuky', '13-Winnti731']

for dataset in validation_datasets:
    try:
        # Load validation events
        val_path = VALIDATION_EVENTS_DIR / f'{dataset}_events.csv'
        
        print(f"\n{'='*80}")
        print(f"Processing: {dataset}")
        print(f"{'='*80}")
        
        val_df = load_and_clean_events(val_path, is_training=False)
        
        # Apply same feature engineering
        val_df = extract_forensic_patterns(val_df)
        val_df = add_cross_artifact_validation(val_df)
        val_df = add_temporal_features(val_df)
        val_df = add_file_characteristics(val_df)
        val_df = parse_timestamp_changes(val_df)
        
        # Save validation features
        output_path = VALIDATION_OUTPUT_DIR / f'{dataset}_features.csv'
        val_df.to_csv(output_path, index=False)
        
        print(f"\nSaved: {output_path}")
        print(f"Shape: {val_df.shape}")
        print(f"Size: {output_path.stat().st_size / 1024 / 1024:.2f} MB")
        
    except Exception as e:
        print(f"ERROR processing {dataset}: {e}")
        import traceback
        traceback.print_exc()
        continue



PROCESSING VALIDATION DATA

Processing: LoneWolf

Loading: LoneWolf_events.csv
  Original shape: (34300, 24)
  Original columns: 24
  Dropped 5 columns
  Final shape: (34300, 16)
  Final columns: 16

Extracting forensic pattern features...
  zero_in_nanoseconds: 309 events
  time_reversal_event: 587 events
  basic_info_changed: 33,713 events
  timestamp_to_past: 0 events
  using_another_timestamp: 0 events

Adding cross-artifact validation features...
  LoneWolf             LF:  437 files | USN: 7416 files | Both:  425 files

  Events with both sources (score=1.0): 3,517
  Events with one source (score=0.5): 30,783

Adding temporal features...
  Calculating 1-minute window counts...
  Calculating 5-minute window counts...
  Max events per file: 2748
  Max events in 1min window: 354
  Max events in 5min window: 354

Adding file characteristic features...
  Executables: 1,108
  Documents: 300
  Images: 5,590
  System files: 3,148

Parsing timestamp changes from Detail field...
  Parsed 

## Feature Summary and Analysis

Review all created features and their statistics.


In [25]:
# Cell 11: Feature Summary

print("\n" + "="*80)
print("PHASE 2 COMPLETE - FEATURE SUMMARY")
print("="*80)

print("\n1. FORENSIC PATTERN FEATURES (11 features):")
print("   - zero_in_nanoseconds (main indicator)")
print("   - zero_in_creation, zero_in_modified, zero_in_mft_modified, zero_in_accessed")
print("   - time_reversal_event")
print("   - basic_info_changed")
print("   - update_resident_value")
print("   - timestamp_to_past")
print("   - using_another_timestamp")

print("\n2. CROSS-ARTIFACT VALIDATION FEATURES (3 features):")
print("   - has_logfile_evidence")
print("   - has_usnjrnl_evidence")
print("   - cross_artifact_validation_score")

print("\n3. TEMPORAL FEATURES (4 features):")
print("   - event_count_per_file")
print("   - events_in_1min_window")
print("   - events_in_5min_window")
print("   - event_datetime")

print("\n4. FILE CHARACTERISTIC FEATURES (8 features):")
print("   - is_executable, is_document, is_image, is_archive")
print("   - is_system_file")
print("   - path_depth")
print("   - filename_length")
print("   - file_extension")

print("\n5. TIMESTAMP CHANGE FEATURES (5 features):")
print("   - timestamp_before, timestamp_after")
print("   - timestamp_before_dt, timestamp_after_dt")
print("   - timestamp_change_seconds")

print("\n" + "="*80)
print("OUTPUT FILES")
print("="*80)
print(f"Training: {OUTPUT_DIR / 'training_features.csv'}")
print(f"Validation: {VALIDATION_OUTPUT_DIR}")
print("\nValidation datasets:")
for dataset in validation_datasets:
    val_path = VALIDATION_OUTPUT_DIR / f'{dataset}_features.csv'
    if val_path.exists():
        print(f"  - {dataset}_features.csv ({val_path.stat().st_size / 1024 / 1024:.2f} MB)")

print("\n" + "="*80)
print("NEXT: Phase 3 - Model Training")
print("="*80)



PHASE 2 COMPLETE - FEATURE SUMMARY

1. FORENSIC PATTERN FEATURES (11 features):
   - zero_in_nanoseconds (main indicator)
   - zero_in_creation, zero_in_modified, zero_in_mft_modified, zero_in_accessed
   - time_reversal_event
   - basic_info_changed
   - update_resident_value
   - timestamp_to_past
   - using_another_timestamp

2. CROSS-ARTIFACT VALIDATION FEATURES (3 features):
   - has_logfile_evidence
   - has_usnjrnl_evidence
   - cross_artifact_validation_score

3. TEMPORAL FEATURES (4 features):
   - event_count_per_file
   - events_in_1min_window
   - events_in_5min_window
   - event_datetime

4. FILE CHARACTERISTIC FEATURES (8 features):
   - is_executable, is_document, is_image, is_archive
   - is_system_file
   - path_depth
   - filename_length
   - file_extension

5. TIMESTAMP CHANGE FEATURES (5 features):
   - timestamp_before, timestamp_after
   - timestamp_before_dt, timestamp_after_dt
   - timestamp_change_seconds

OUTPUT FILES
Training: /Users/soni/Github/Digital-Dete

## Feature Correlation Analysis

Analyze which features correlate most strongly with the suspicious label (training data only).

This helps validate that our features are capturing timestomping patterns.


In [26]:
# Cell 12: Feature Correlation Analysis

print("\n" + "="*80)
print("FEATURE CORRELATION WITH SUSPICIOUS LABEL")
print("="*80)

if 'is_suspicious' in training_df.columns:
    print("\nTraining data statistics:")
    print(f"  Total events: {len(training_df):,}")
    print(f"  Suspicious events: {training_df['is_suspicious'].sum():,} ({training_df['is_suspicious'].mean()*100:.3f}%)")
    print(f"  Benign events: {(~training_df['is_suspicious']).sum():,}")
    
    print("\nTop forensic features activated in suspicious events:")
    
    feature_cols = [
        'zero_in_nanoseconds',
        'zero_in_creation',
        'zero_in_modified',
        'time_reversal_event',
        'basic_info_changed',
        'timestamp_to_past',
        'using_another_timestamp',
        'cross_artifact_validation_score',
        'event_count_per_file',
        'is_executable'
    ]
    
    print(f"\n{'Feature':<40} {'Correlation':>12} {'In Suspicious':>15} {'In Benign':>12}")
    print("="*80)
    
    for col in feature_cols:
        if col in training_df.columns:
            # Calculate correlation
            if training_df[col].dtype == 'bool':
                corr = training_df[[col, 'is_suspicious']].astype(int).corr().iloc[0, 1]
                susp_pct = training_df[training_df['is_suspicious']][col].mean() * 100
                benign_pct = training_df[~training_df['is_suspicious']][col].mean() * 100
            else:
                corr = training_df[[col, 'is_suspicious']].corr().iloc[0, 1]
                susp_mean = training_df[training_df['is_suspicious']][col].mean()
                benign_mean = training_df[~training_df['is_suspicious']][col].mean()
                susp_pct = susp_mean
                benign_pct = benign_mean
            
            if training_df[col].dtype == 'bool':
                print(f"{col:<40} {corr:>12.3f} {susp_pct:>14.1f}% {benign_pct:>11.1f}%")
            else:
                print(f"{col:<40} {corr:>12.3f} {susp_pct:>14.2f} {benign_pct:>11.2f}")
    
    print("\nInterpretation:")
    print("  - Positive correlation: Feature is more common in suspicious events")
    print("  - High % in suspicious, low % in benign: Strong discriminative feature")
    print("  - Cross-artifact score ~1.0 in suspicious: Files detected by both sources")

else:
    print("\nNo ground truth labels in this dataset (validation mode)")



FEATURE CORRELATION WITH SUSPICIOUS LABEL

Training data statistics:
  Total events: 373,114
  Suspicious events: 288 (0.077%)
  Benign events: 372,826

Top forensic features activated in suspicious events:

Feature                                   Correlation   In Suspicious    In Benign
zero_in_nanoseconds                             0.007            2.8%         0.7%
zero_in_creation                                0.011            2.4%         0.3%
zero_in_modified                                0.006            2.4%         0.6%
time_reversal_event                             0.017           10.1%         1.8%
basic_info_changed                             -0.019           88.9%        98.2%
timestamp_to_past                                 nan            0.0%         0.0%
using_another_timestamp                           nan            0.0%         0.0%
cross_artifact_validation_score                 0.070           1.00        0.57
event_count_per_file                          